# Randomness You Can Trust (QRNG)

*Warm-up · UTC Problem 5*

An encryption key is only as safe as the random numbers behind it. Make a key two ways, with software random numbers and with a quantum circuit, then watch an attacker try to copy each one.

> **Before you run anything, pick the kernel.** When this notebook opens, a **Select Kernel** box appears with *Python 3 (ipykernel)* already selected. That's the wrong one: change it to **qBraid-SDK (v0.12.2)**, then click **Select**. Picked the wrong one, or closed the box? Click the kernel name at the top right of the notebook and choose **qBraid-SDK (v0.12.2)**. The first cell checks, and tells you if it's wrong.

In [ ]:
# Use the qBraid-SDK (v0.12.2) kernel. This stops with a clear message if another one is picked.
import importlib.util
missing = [m for m in ("qbraid", "matplotlib") if importlib.util.find_spec(m) is None]
if missing:
    raise ImportError("Wrong kernel: this one has no " + " or ".join(missing) + ". Click the kernel name "
                      "at the top right of the notebook, pick qBraid-SDK (v0.12.2), then run this cell again.")

# Qiskit Aer is Qiskit's simulator. It runs right here in your Lab instance: instant, free and noise-free,
# with no job and no credits. The first run installs it (a few seconds), pinned to the version tested here.
try:
    import qiskit_aer  # noqa: F401
except ImportError:
    %pip install -q qiskit-aer==0.18.2

import random, time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
sim = AerSimulator()

def draw(rows):
    """Each key next to the attacker's copy of it, with the attacker's wrong bits in red."""
    colors = ListedColormap(["#e9ecef", "#1b1f3b", "#e63946"])   # a 0, a 1, a wrong bit
    fig, axes = plt.subplots(len(rows), 2, figsize=(8, 9))
    for (title, key, copy, verdict, cracked), (left, right) in zip(rows, axes):
        k = np.array(list(key), dtype=int).reshape(16, 16)
        c = np.array(list(copy), dtype=int).reshape(16, 16)
        left.imshow(k, cmap=colors, vmin=0, vmax=2)
        left.set_title(f"{title}\n{key.count('1') / 256:.0%} ones: looks random", fontsize=12)
        right.imshow(np.where(c == k, c, 2), cmap=colors, vmin=0, vmax=2)
        right.set_title(f"The attacker's copy\n{verdict}", fontsize=12, fontweight="bold",
                        color="#c1121f" if cracked else "#2a9d8f")
        for ax in (left, right):
            ax.set_xticks(np.arange(-0.5, 16), minor=True)
            ax.set_yticks(np.arange(-0.5, 16), minor=True)
            ax.grid(which="minor", color="white", linewidth=0.8)
            ax.tick_params(which="both", length=0, labelbottom=False, labelleft=False)
    fig.text(0.5, 0.01, "Each square is one bit of the key. Red: a bit the attacker got wrong.",
             ha="center", fontsize=11)
    fig.tight_layout(rect=(0, 0.03, 1, 1), h_pad=3)
    plt.show()

print("Setup OK: the qBraid kernel and Qiskit Aer are ready. You're good to go.")

In [ ]:
def qrng_circuit(n):
    """A circuit whose measurements are random bits."""
    qc = QuantumCircuit(n, n)
    # TODO: put every qubit in superposition, then measure.
    # Hint: qc.h(range(n)); qc.measure(range(n), range(n))
    # When your two lines are in, delete the next line.
    raise NotImplementedError("The TODO in qrng_circuit (the cell above) isn't done yet: add the two lines from the hint, delete this raise line, run that cell again, then rerun this one.")
    return qc


### Make a key two ways, then attack it

A grid device, say a smart meter, makes a 256-bit encryption key when it starts up. The next cell makes that key two ways: with software random numbers seeded with the clock, a common shortcut, and with your quantum circuit. Then an attacker who knows the device started sometime in the last 24 hours tries every second of that window as the seed.

In [ ]:
# A grid device makes a 256-bit key when it starts up. Here it does that two ways.
now = int(time.time())

# 1) Software random numbers, seeded with the clock (a common shortcut)
started = now - random.randrange(86_400)   # it started sometime in the last 24 hours
software_key = format(random.Random(started).getrandbits(256), "0256b")

# 2) Your quantum circuit: 32 shots of 8 qubits = 256 bits, in the order measured
qc = qrng_circuit(8)
if "measure" not in qc.count_ops():
    raise ValueError("qrng_circuit doesn't measure yet: check it against the hint.")
result = sim.run(transpile(qc, sim), shots=32, memory=True).result()
quantum_key = "".join(result.get_memory())
if len(quantum_key) != 256 or not 0.3 < quantum_key.count("1") / 256 < 0.7:
    raise ValueError("Your circuit's bits don't look random yet. "
                     "Check qrng_circuit against the hint.")

# The attacker knows when it could have started, so they try every second as the seed
def attack(key):
    """Returns the attacker's copy and the seed behind it (None if no seed works)."""
    target = int(key, 2)
    for seed in range(now, now - 86_400, -1):
        if random.Random(seed).getrandbits(256) == target:
            return format(random.Random(seed).getrandbits(256), "0256b"), seed
    # no seed works, so the attacker can only guess
    return format(random.getrandbits(256), "0256b"), None

rows = []
for title, key in [("Software key, seeded with the clock", software_key),
                   ("Quantum key, your circuit (simulated)", quantum_key)]:
    t0 = time.time()
    copy, seed = attack(key)
    secs = time.time() - t0
    right = sum(a == b for a, b in zip(copy, key))
    if seed is not None:
        ago = now - seed
        print(f"Software key: CRACKED in {secs:.1f} s. The seed was the clock when the "
              f"device started, {ago // 3600} h {ago % 3600 // 60} min ago.")
        verdict = f"CRACKED: {right} of 256 bits right"
    else:
        print(f"Quantum key: none of the 86,400 seeds works, so the attacker can only "
              f"guess: {right} of 256 bits right, the same as flipping a coin.")
        verdict = f"NOT CRACKED: {right} of 256 bits right"
    rows.append((title, key, copy, verdict, seed is not None))

draw(rows)
aer_seed = result.results[0].seed_simulator
print(f"Fine print: Aer is a simulator, so it's software too. It made your quantum key "
      f"from a hidden seed, {aer_seed}, picked fresh each run instead of from the clock: "
      "that's why the attack missed. Anyone with that number can replay the key. "
      "A real quantum computer has no seed at all.")

### Run it through qBraid

Run the cell below as written: it sends your circuit to IonQ's **free** simulator and prints a job ID. Find that job on the [Jobs page](https://account.qbraid.com/quantum-jobs) in your qBraid account. If it's there, your account works.

Optional, after that: swap in a real quantum computer, the only kind with no seed at all. It uses credits, so cap it at 200 shots.

In [ ]:
# --- Run on a quantum computer through qBraid ---
from qbraid.runtime import QbraidProvider
import warnings
# IonQ measures every qubit anyway, so skip its warning about measurement gates
warnings.filterwarnings("ignore", message="Circuit contains measurement gates")

provider = QbraidProvider()   # in qBraid Lab this uses your account automatically

# 1) Free: IonQ's simulator. It's software too, with an approximate model of IonQ's noise
#    (this circuit has no two-qubit gates, so it's nearly noise-free).
device = provider.get_device("ionq:ionq:sim:simulator")
job = device.run(qc, shots=200, runtime_options={"noise_model": "aria-1"})

# 1b) Also free: Quantinuum's H2 emulator, which models a real trapped-ion computer's noise.
#     Accept Quantinuum's terms once first (Devices > Quantinuum in your qBraid account):
# device = provider.get_device("quantinuum:quantinuum:sim:h2-emulator")
# job = device.run(qc, shots=200)

# 2) Optional: a real quantum computer (uses credits). Swap in ONE device and rerun.
#    Prices for 200 shots, from the device catalog on Nov 11 (check it for live status):
#    Rigetti Cepheus about 40 credits, IQM Garnet about 60, IonQ Forte about 1,630.
# device = provider.get_device("aws:rigetti:qpu:cepheus-1-108q")
# device = provider.get_device("aws:iqm:qpu:garnet")
# device = provider.get_device("aws:ionq:qpu:forte-1")
# job = device.run(qc, shots=200)

print("Job ID:", job.id)          # find this job on the Jobs page in your qBraid account
counts = job.result().data.get_counts()
bits = sum(len(b) * c for b, c in counts.items())
ones = sum(b.count("1") * c for b, c in counts.items()) / bits
print(f"{sum(counts.values())} shots came back, and {ones:.0%} of their bits are 1s.")

## Why it matters

- **Software random numbers are a recipe.** The same seed always gives the same bits, so anyone who guesses or steals the seed can remake the key. It really happens: in 1995, two Berkeley students broke the encryption in Netscape's web browser, usually in under a minute, because it seeded its random numbers with the time of day and two process numbers.
- **Good software does better.** It takes its seed from the operating system instead of the clock, and this attack fails. But there's still a seed to steal. A measurement on a real quantum computer has none.
- **It's a business.** Quantinuum, a hackathon sponsor, sells encryption keys seeded by its quantum computers (Quantum Origin). In 2025, Quantinuum and JPMorganChase, working with Oak Ridge National Laboratory and others, used Quantinuum's H2 computer to make the first certified random numbers, with supercomputers checking that they really came from the quantum computer.

## If you have time (nothing to submit)

- Rerun the attack cell: both keys change every time. Now copy the hidden seed it printed into `sim = AerSimulator(seed_simulator=...)`, run that, and rerun the attack cell: the same quantum key comes back, square for square. That's the software underneath.
- Run the qBraid cell on Quantinuum's free H2 emulator, or on a real quantum computer (Rigetti through AWS is the cheapest, about 40 credits).
- In a sentence or two: why would a utility managing thousands of device keys care?